# Lab 06: Bedrock Agents & Tool Use

## Overview
Create a Bedrock Agent with Lambda action groups, OpenAPI schemas, and ReAct orchestration. Implement function calling and return-of-control patterns.

## Learning Objectives
- Create and configure a Bedrock Agent with foundation model
- Define Lambda action groups with OpenAPI schemas
- Understand ReAct orchestration pattern
- Implement return of control for application-managed actions
- Manage agent sessions for multi-turn conversations

## Exam Domain
**Building Generative AI Applications (30%)** — covers Task 2.2 (implement agents and tool use).

## Architecture Diagram
![Lab 06 Architecture](../assets/diagrams/lab-06-bedrock-agents.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3, json, os, time, zipfile, io

REGION = "us-east-1"
session = boto3.Session(region_name=REGION)
bedrock_agent_client = session.client("bedrock-agent")
bedrock_agent_runtime = session.client("bedrock-agent-runtime")
lambda_client = session.client("lambda")
iam_client = session.client("iam")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-lambda-role"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"
print(f"Account: {ACCOUNT_ID}")

## A. Create Lambda Function for Action Group

A **Bedrock Agent** uses **action groups** to perform tasks. Each action group maps to a Lambda function that executes the actual business logic. When the agent decides it needs to call a tool, it invokes the Lambda function with a structured event containing the API path, HTTP method, and parameters extracted from the user's query.

Our Lambda function implements two endpoints:

1. **`/lookup-service`** — Returns details about a specific AWS service (description, category, pricing model)
2. **`/compare-services`** — Compares two AWS services side-by-side

This simulates a real-world scenario where an agent has access to a service catalog API. The agent uses ReAct (Reasoning + Acting) orchestration to decide which endpoint to call based on the user's question.

In [ ]:
# Define the Lambda function code
lambda_code = """
import json

# Simulated AWS service catalog
SERVICE_CATALOG = {
    "bedrock": {
        "name": "Amazon Bedrock",
        "category": "AI/ML",
        "description": "Fully managed service to build generative AI applications with foundation models.",
        "pricing_model": "On-demand per token/image",
        "key_features": ["Foundation model access", "Knowledge Bases", "Agents", "Guardrails", "Model customization"]
    },
    "sagemaker": {
        "name": "Amazon SageMaker",
        "category": "AI/ML",
        "description": "Fully managed service to build, train, and deploy machine learning models at scale.",
        "pricing_model": "Per instance-hour",
        "key_features": ["Notebooks", "Training jobs", "Endpoints", "Pipelines", "Feature Store"]
    },
    "lambda": {
        "name": "AWS Lambda",
        "category": "Compute",
        "description": "Serverless compute service that runs code in response to events.",
        "pricing_model": "Per request + duration",
        "key_features": ["Event-driven", "Auto-scaling", "Multiple runtimes", "Layers", "Container support"]
    },
    "s3": {
        "name": "Amazon S3",
        "category": "Storage",
        "description": "Object storage service offering industry-leading scalability and durability.",
        "pricing_model": "Per GB stored + requests",
        "key_features": ["11 nines durability", "Storage classes", "Versioning", "Lifecycle policies", "Event notifications"]
    },
    "dynamodb": {
        "name": "Amazon DynamoDB",
        "category": "Database",
        "description": "Fully managed NoSQL database service providing fast and predictable performance.",
        "pricing_model": "Per RCU/WCU or on-demand",
        "key_features": ["Single-digit ms latency", "Auto-scaling", "Global tables", "Streams", "DAX caching"]
    }
}


def handler(event, context):
    api_path = event.get("apiPath", "")
    parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

    if api_path == "/lookup-service":
        service_id = parameters.get("service_id", "").lower()
        service = SERVICE_CATALOG.get(service_id)
        if service:
            body = json.dumps(service)
        else:
            body = json.dumps({"error": f"Service '{service_id}' not found. Available: {list(SERVICE_CATALOG.keys())}"})

    elif api_path == "/compare-services":
        svc1_id = parameters.get("service_id_1", "").lower()
        svc2_id = parameters.get("service_id_2", "").lower()
        svc1 = SERVICE_CATALOG.get(svc1_id)
        svc2 = SERVICE_CATALOG.get(svc2_id)
        if svc1 and svc2:
            body = json.dumps({
                "comparison": {
                    svc1_id: {"name": svc1["name"], "category": svc1["category"], "pricing": svc1["pricing_model"], "features": svc1["key_features"]},
                    svc2_id: {"name": svc2["name"], "category": svc2["category"], "pricing": svc2["pricing_model"], "features": svc2["key_features"]}
                }
            })
        else:
            missing = [s for s, v in [(svc1_id, svc1), (svc2_id, svc2)] if not v]
            body = json.dumps({"error": f"Service(s) not found: {missing}. Available: {list(SERVICE_CATALOG.keys())}"})
    else:
        body = json.dumps({"error": f"Unknown API path: {api_path}"})

    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup", ""),
            "apiPath": api_path,
            "httpMethod": event.get("httpMethod", "GET"),
            "httpStatusCode": 200,
            "responseBody": {"application/json": {"body": body}}
        }
    }
"""

print("Lambda function code defined")
print(f"Code length: {len(lambda_code)} characters")

### Understanding the Lambda Response Format

The Lambda function must return responses in Bedrock's expected format. The key fields are:

| Field | Purpose |
|-------|---------|
| `messageVersion` | Must be `"1.0"` — tells Bedrock which response schema to expect |
| `response.actionGroup` | Echoes back the action group name from the incoming event |
| `response.apiPath` | Echoes back the API path that was invoked |
| `response.httpMethod` | The HTTP method (GET, POST, etc.) |
| `response.httpStatusCode` | Status code — 200 for success |
| `response.responseBody` | The actual response data, keyed by content type |

If the Lambda returns a response that does not match this schema, the agent will fail to parse the tool result and return an error to the user.

### Package and Deploy the Lambda Function

We package the code as a zip file in memory and deploy it using the Lambda `create_function` API. After creation, we add a **resource-based policy** that grants the Bedrock service permission to invoke the function. Without this policy, the agent would receive an "Access Denied" error when trying to call the Lambda.

In [ ]:
LAMBDA_NAME = "genai-lab-agent-action"

# Package code as zip in memory
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", lambda_code)
zip_buffer.seek(0)

# Create the Lambda function
try:
    fn = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.handler",
        Code={"ZipFile": zip_buffer.read()},
        Timeout=30,
        MemorySize=128,
        Description="Action group handler for Bedrock Agent - AWS service catalog"
    )
    LAMBDA_ARN = fn["FunctionArn"]
    print(f"Lambda created: {LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"
    print(f"Lambda already exists: {LAMBDA_ARN}")

# Wait for function to become Active
print("Waiting for function to become Active...")
waiter = lambda_client.get_waiter("function_active_v2")
waiter.wait(FunctionName=LAMBDA_NAME)
print("Lambda function is Active")

In [ ]:
# Grant Bedrock permission to invoke the Lambda function
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="AllowBedrockInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock.amazonaws.com",
        SourceArn=f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:agent/*"
    )
    print("Resource-based policy added — Bedrock can now invoke the Lambda")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists")

## B. Define OpenAPI Schema

The **OpenAPI schema** tells the Bedrock Agent what tools are available, what parameters they accept, and what they return. The agent uses this schema to:

1. **Decide which tool to call** — the operation descriptions help the model understand when each endpoint is relevant
2. **Extract parameters** — the parameter definitions tell the model what information to extract from the user's query
3. **Validate requests** — Bedrock ensures the agent provides all required parameters before invoking Lambda

The schema must follow the OpenAPI 3.0 specification. Each path/operation maps to one tool the agent can use.

In [ ]:
openapi_schema = {
    "openapi": "3.0.0",
    "info": {
        "title": "AWS Service Catalog API",
        "version": "1.0.0",
        "description": "API for looking up and comparing AWS services"
    },
    "paths": {
        "/lookup-service": {
            "get": {
                "summary": "Look up details about an AWS service",
                "description": "Returns the name, category, description, pricing model, and key features for a specific AWS service. Use this when the user asks about a single service.",
                "operationId": "lookupService",
                "parameters": [
                    {
                        "name": "service_id",
                        "in": "query",
                        "description": "The identifier of the AWS service to look up (e.g., bedrock, sagemaker, lambda, s3, dynamodb)",
                        "required": True,
                        "schema": {"type": "string"}
                    }
                ],
                "responses": {
                    "200": {
                        "description": "Service details returned successfully",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "object",
                                    "properties": {
                                        "name": {"type": "string"},
                                        "category": {"type": "string"},
                                        "description": {"type": "string"},
                                        "pricing_model": {"type": "string"},
                                        "key_features": {"type": "array", "items": {"type": "string"}}
                                    }
                                }
                            }
                        }
                    }
                }
            }
        },
        "/compare-services": {
            "get": {
                "summary": "Compare two AWS services side by side",
                "description": "Returns a comparison of two AWS services including their categories, pricing models, and key features. Use this when the user wants to compare two services.",
                "operationId": "compareServices",
                "parameters": [
                    {
                        "name": "service_id_1",
                        "in": "query",
                        "description": "The identifier of the first AWS service (e.g., bedrock, sagemaker, lambda, s3, dynamodb)",
                        "required": True,
                        "schema": {"type": "string"}
                    },
                    {
                        "name": "service_id_2",
                        "in": "query",
                        "description": "The identifier of the second AWS service (e.g., bedrock, sagemaker, lambda, s3, dynamodb)",
                        "required": True,
                        "schema": {"type": "string"}
                    }
                ],
                "responses": {
                    "200": {
                        "description": "Service comparison returned successfully",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "object",
                                    "properties": {
                                        "comparison": {"type": "object"}
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
    }
}

print("OpenAPI Schema defined with 2 endpoints:")
for path, methods in openapi_schema["paths"].items():
    for method, details in methods.items():
        params = [p["name"] for p in details.get("parameters", [])]
        print(f"  {method.upper()} {path} — params: {params}")

## C. Create the Bedrock Agent

Creating a Bedrock Agent involves four steps:

1. **Create the agent** — define its name, foundation model, and instructions
2. **Add an action group** — connect the OpenAPI schema and Lambda function
3. **Prepare the agent** — compile the agent configuration into a deployable state
4. **Create an alias** — create a versioned endpoint for invoking the agent

The **agent instructions** are critical. They serve as the system prompt that guides the agent's behavior — how it interprets user questions, when to use tools, and how to format responses. Well-crafted instructions significantly improve tool selection accuracy.

### ReAct Orchestration

Bedrock Agents use **ReAct (Reasoning + Acting)** orchestration by default. In each turn, the agent:
1. **Thinks** — reasons about the user's request and what information it needs
2. **Acts** — calls a tool (action group) to get information
3. **Observes** — processes the tool's response
4. **Repeats** — continues thinking/acting until it can provide a final answer

This loop allows agents to handle complex queries that require multiple tool calls or multi-step reasoning.

In [ ]:
AGENT_NAME = "genai-lab-service-agent"
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Step 1: Create the agent
agent = bedrock_agent_client.create_agent(
    agentName=AGENT_NAME,
    agentResourceRoleArn=BEDROCK_ROLE_ARN,
    foundationModel=MODEL_ID,
    instruction="""You are an AWS service catalog assistant. Your job is to help users
understand AWS services by looking up details and comparing services.

When a user asks about a specific AWS service, use the lookupService tool to get its details.
When a user wants to compare two services, use the compareServices tool.
Available services: bedrock, sagemaker, lambda, s3, dynamodb.

Always provide clear, structured responses based on the data returned by the tools.
If a service is not in the catalog, let the user know which services are available.""",
    idleSessionTTLInSeconds=600
)

AGENT_ID = agent["agent"]["agentId"]
print(f"Agent created: {AGENT_ID}")
print(f"Name:          {agent['agent']['agentName']}")
print(f"Status:        {agent['agent']['agentStatus']}")
print(f"Model:         {MODEL_ID}")

In [ ]:
# Step 2: Add action group with OpenAPI schema and Lambda
action_group = bedrock_agent_client.create_agent_action_group(
    agentId=AGENT_ID,
    agentVersion="DRAFT",
    actionGroupName="ServiceCatalogActions",
    actionGroupExecutor={"lambda": LAMBDA_ARN},
    apiSchema={"payload": json.dumps(openapi_schema)},
    description="Actions for looking up and comparing AWS services in the catalog"
)

AG_ID = action_group["agentActionGroup"]["actionGroupId"]
print(f"Action group created: {AG_ID}")
print(f"Name: {action_group['agentActionGroup']['actionGroupName']}")

In [ ]:
# Step 3: Prepare the agent (compiles configuration into a deployable version)
bedrock_agent_client.prepare_agent(agentId=AGENT_ID)
print("Preparing agent...")

# Poll until preparation completes
while True:
    status = bedrock_agent_client.get_agent(agentId=AGENT_ID)
    agent_status = status["agent"]["agentStatus"]
    print(f"  Status: {agent_status}")
    if agent_status in ("PREPARED", "FAILED", "NOT_PREPARED"):
        break
    time.sleep(5)

if agent_status == "PREPARED":
    print("\nAgent is ready!")
else:
    print(f"\nAgent preparation failed: {agent_status}")

In [ ]:
# Step 4: Create an alias (required for invoking the agent)
alias = bedrock_agent_client.create_agent_alias(
    agentId=AGENT_ID,
    agentAliasName="live"
)

AGENT_ALIAS_ID = alias["agentAlias"]["agentAliasId"]
print(f"Agent alias created: {AGENT_ALIAS_ID}")

# Wait for alias to become active
time.sleep(5)
alias_status = bedrock_agent_client.get_agent_alias(
    agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID
)
print(f"Alias status: {alias_status['agentAlias']['agentAliasStatus']}")

## D. Invoke the Agent

Now we invoke the agent with natural language queries. The agent will use ReAct orchestration to:
1. Parse the user's question
2. Decide which tool to call (lookup or compare)
3. Extract the required parameters
4. Invoke the Lambda function
5. Synthesize the tool's response into a natural language answer

The `invoke_agent` API returns a streaming response. We define a helper function to collect the stream into a complete response string.

### Session Management

Each invocation includes a `sessionId`. Using the **same session ID** across multiple calls enables multi-turn conversations — the agent remembers previous interactions within the session. Using a **different session ID** starts a fresh conversation with no history.

In [ ]:
import uuid

def invoke_agent(query, session_id=None, enable_trace=False):
    """Invoke the Bedrock Agent and return the response text."""
    if session_id is None:
        session_id = str(uuid.uuid4())

    response = bedrock_agent_runtime.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id,
        inputText=query,
        enableTrace=enable_trace
    )

    # Collect streaming response
    full_response = ""
    for event in response["completion"]:
        if "chunk" in event:
            chunk_text = event["chunk"]["bytes"].decode("utf-8")
            full_response += chunk_text
        if enable_trace and "trace" in event:
            trace = event["trace"].get("trace", {})
            if "orchestrationTrace" in trace:
                orch = trace["orchestrationTrace"]
                if "rationale" in orch:
                    print(f"  [Thought] {orch['rationale']['text'][:200]}")
                if "invocationInput" in orch:
                    inv = orch["invocationInput"]
                    if "actionGroupInvocationInput" in inv:
                        ag = inv["actionGroupInvocationInput"]
                        print(f"  [Action]  {ag.get('apiPath', 'N/A')} — params: {ag.get('parameters', [])}")

    return full_response, session_id

print("Helper function defined")

In [ ]:
# Query 1: Single service lookup (with trace enabled to see ReAct reasoning)
print("Query: 'Tell me about Amazon Bedrock'\n")

response_text, session_id = invoke_agent(
    "Tell me about Amazon Bedrock",
    enable_trace=True
)
print(f"\nResponse:\n{response_text}")

In [ ]:
# Query 2: Service comparison
print("Query: 'Compare Bedrock and SageMaker'\n")

response_text, _ = invoke_agent(
    "Compare Bedrock and SageMaker",
    enable_trace=True
)
print(f"\nResponse:\n{response_text}")

### Multi-Turn Conversation

By reusing the same `sessionId`, the agent maintains conversation context. In this example, we first ask about Lambda, then ask a follow-up question that references "it" — the agent resolves the pronoun using session history.

In [ ]:
# Multi-turn: Turn 1 — ask about Lambda
session_id = str(uuid.uuid4())
print("Turn 1: 'What is AWS Lambda?'\n")
response_1, session_id = invoke_agent("What is AWS Lambda?", session_id=session_id)
print(f"Response:\n{response_1}")

print("\n" + "=" * 60 + "\n")

# Multi-turn: Turn 2 — follow-up using same session (agent remembers context)
print("Turn 2: 'How does its pricing compare to S3?'\n")
response_2, session_id = invoke_agent(
    "How does its pricing compare to S3?",
    session_id=session_id
)
print(f"Response:\n{response_2}")

## E. Return of Control Pattern

In the standard flow above, the agent invokes Lambda directly to execute tools. **Return of Control (ROC)** is an alternative pattern where the agent returns control to your application instead of calling Lambda. Your application executes the action, then sends the result back to the agent to continue processing.

### When to Use Return of Control

| Pattern | Use Case |
|---------|----------|
| **Lambda Executor** | Stateless API calls, database lookups, third-party integrations — anything that can run independently |
| **Return of Control** | Actions requiring client-side context (user's browser, local files), human approval workflows, actions that need application state not available in Lambda |

### How It Works

1. The agent decides to call a tool (same as before)
2. Instead of invoking Lambda, Bedrock returns an `invocationInput` to your application with the action group name, API path, and parameters
3. Your application executes the action using its own logic
4. Your application calls `invoke_agent` again with the result in `sessionState.returnControlInvocationResults`
5. The agent continues reasoning with the tool result

This pattern is configured by setting `actionGroupExecutor` to `{"customControl": "RETURN_CONTROL"}` instead of providing a Lambda ARN.

In [ ]:
# Conceptual example: Return of Control pattern
# This demonstrates the code structure — we do not create a separate agent for this

roc_example = """
# --- Creating an action group with Return of Control ---
bedrock_agent_client.create_agent_action_group(
    agentId=AGENT_ID,
    agentVersion="DRAFT",
    actionGroupName="ClientSideActions",
    actionGroupExecutor={"customControl": "RETURN_CONTROL"},  # <-- Key difference
    apiSchema={"payload": json.dumps(openapi_schema)},
    description="Actions executed by the calling application"
)

# --- Invoking and handling Return of Control ---
response = bedrock_agent_runtime.invoke_agent(
    agentId=AGENT_ID,
    agentAliasId=AGENT_ALIAS_ID,
    sessionId=session_id,
    inputText="Tell me about Bedrock"
)

for event in response["completion"]:
    if "returnControl" in event:
        # Agent is requesting your app to execute an action
        invocation = event["returnControl"]["invocationInputs"][0]
        api_input = invocation["apiInvocationInput"]
        api_path = api_input["apiPath"]
        parameters = api_input.get("parameters", [])
        invocation_id = event["returnControl"]["invocationId"]

        print(f"Agent wants to call: {api_path}")
        print(f"With parameters: {parameters}")

        # YOUR APPLICATION executes the action here
        result = your_local_function(api_path, parameters)

        # Send the result back to the agent
        followup = bedrock_agent_runtime.invoke_agent(
            agentId=AGENT_ID,
            agentAliasId=AGENT_ALIAS_ID,
            sessionId=session_id,
            sessionState={
                "invocationId": invocation_id,
                "returnControlInvocationResults": [{
                    "apiResult": {
                        "actionGroup": "ClientSideActions",
                        "apiPath": api_path,
                        "httpMethod": "GET",
                        "httpStatusCode": 200,
                        "responseBody": {
                            "application/json": {"body": json.dumps(result)}
                        }
                    }
                }]
            }
        )
        # Process the agent's final response from followup...
"""

print("Return of Control — Conceptual Code Pattern")
print("=" * 50)
print(roc_example)

## F. Appendix — LangChain ReAct Agent (Optional)

> **This section is optional.** It demonstrates building a ReAct agent using LangChain and ChatBedrock with custom tool definitions. This is an alternative to Bedrock's managed Agents service — useful when you need more control over the orchestration loop or want to run agents locally without creating AWS resources.

With LangChain, you define tools as Python functions and the framework handles the ReAct loop (think-act-observe) using the foundation model's native function calling capability.

In [ ]:
%pip install langchain-aws langgraph --quiet

In [ ]:
# Optional: LangChain ReAct agent with tool definitions
from langchain_aws import ChatBedrock
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Define tools as Python functions
@tool
def lookup_service(service_id: str) -> str:
    """Look up details about an AWS service. Available services: bedrock, sagemaker, lambda, s3, dynamodb."""
    catalog = {
        "bedrock": "Amazon Bedrock — AI/ML — Fully managed generative AI service. Pricing: per token. Features: FM access, Knowledge Bases, Agents, Guardrails.",
        "sagemaker": "Amazon SageMaker — AI/ML — Build, train, deploy ML models. Pricing: per instance-hour. Features: Notebooks, Training, Endpoints, Pipelines.",
        "lambda": "AWS Lambda — Compute — Serverless compute. Pricing: per request + duration. Features: Event-driven, auto-scaling, multiple runtimes.",
        "s3": "Amazon S3 — Storage — Object storage. Pricing: per GB + requests. Features: 11 nines durability, storage classes, versioning.",
        "dynamodb": "Amazon DynamoDB — Database — Managed NoSQL. Pricing: per RCU/WCU. Features: Single-digit ms latency, global tables, streams."
    }
    return catalog.get(service_id.lower(), f"Service '{service_id}' not found. Available: {list(catalog.keys())}")

@tool
def compare_services(service_id_1: str, service_id_2: str) -> str:
    """Compare two AWS services side by side."""
    result_1 = lookup_service.invoke({"service_id": service_id_1})
    result_2 = lookup_service.invoke({"service_id": service_id_2})
    return f"Service 1: {result_1}\n\nService 2: {result_2}"

# Create the LangChain ReAct agent
llm = ChatBedrock(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=REGION
)

agent = create_react_agent(llm, [lookup_service, compare_services])

# Invoke the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Amazon S3 and how does it compare to DynamoDB?"}]}
)

# Print the final response
print("LangChain Agent Response:")
print(result["messages"][-1].content)

## Cleanup

Delete the agent alias, the agent, and the Lambda function. The IAM roles are shared infrastructure managed by `setup-resources.py` and should not be deleted here.

In [ ]:
# Delete agent alias
try:
    bedrock_agent_client.delete_agent_alias(agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID)
    print(f"Deleted agent alias: {AGENT_ALIAS_ID}")
except Exception as e:
    print(f"Could not delete alias: {e}")

time.sleep(3)

# Delete agent (this also deletes associated action groups)
try:
    bedrock_agent_client.delete_agent(agentId=AGENT_ID)
    print(f"Deleted agent: {AGENT_ID}")
except Exception as e:
    print(f"Could not delete agent: {e}")

# Delete Lambda function
try:
    lambda_client.delete_function(FunctionName=LAMBDA_NAME)
    print(f"Deleted Lambda function: {LAMBDA_NAME}")
except Exception as e:
    print(f"Could not delete Lambda: {e}")

print("\nAll lab resources cleaned up.")

## Key Takeaways

1. **Bedrock Agents automate multi-step reasoning** — the ReAct orchestration loop (think-act-observe) lets agents break complex questions into tool calls without custom orchestration code
2. **OpenAPI schemas define the tool contract** — the schema descriptions directly influence tool selection accuracy; clear, specific descriptions help the model choose the right tool and extract correct parameters
3. **Lambda action groups provide serverless execution** — the agent invokes Lambda functions on demand with no infrastructure to manage, and the resource-based policy controls access
4. **Return of Control enables client-side execution** — when actions require local context, human approval, or application state, ROC returns control to your code instead of invoking Lambda
5. **Session management enables multi-turn conversations** — reusing a `sessionId` across invocations gives the agent memory of previous interactions, enabling follow-up questions and context resolution

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Agent | A Bedrock resource that uses a foundation model with ReAct orchestration to reason about user requests, decide which tools to call, and synthesize responses from tool outputs |
| Action Group | A collection of tools (API operations) available to an agent, defined by an OpenAPI schema and backed by either a Lambda function or return-of-control execution |
| OpenAPI Schema | A JSON/YAML specification (OpenAPI 3.0) that describes the available API endpoints, parameters, and response formats — the agent uses this to understand what tools exist and how to call them |
| ReAct | Reasoning + Acting — an orchestration pattern where the agent iterates through thinking (reasoning about what to do), acting (calling a tool), and observing (processing the result) until it can provide a final answer |
| Return of Control | An execution pattern where the agent returns tool invocation details to the calling application instead of executing Lambda directly, allowing client-side execution and human-in-the-loop workflows |
| Session Management | The mechanism for maintaining conversation state across multiple agent invocations using a shared `sessionId`, enabling multi-turn dialogues with context retention |
| Lambda Executor | The default action group execution mode where Bedrock directly invokes a Lambda function to execute the tool, passing the API path and extracted parameters in a structured event |

## Exam Preparation — Common Exam Question Patterns

**Q: What orchestration strategy do Bedrock Agents use, and how does it work?**
A: Bedrock Agents use ReAct (Reasoning + Acting) orchestration. The agent iterates through a loop of thinking (analyzing the user's request), acting (calling a tool via an action group), and observing (processing the tool's response). This continues until the agent has enough information to provide a final answer. The foundation model drives the reasoning, while action groups provide the tools.

**Q: What is the difference between Lambda executor and Return of Control in Bedrock Agents?**
A: With Lambda executor, the agent directly invokes a Lambda function to execute actions — Bedrock manages the entire flow. With Return of Control, the agent returns the action details (API path, parameters) to the calling application, which executes the action using its own logic and sends results back. Use ROC when actions require client-side context, human approval, or application state not available in Lambda.

**Q: What role does the OpenAPI schema play in a Bedrock Agent action group?**
A: The OpenAPI schema defines the tools available to the agent — the API endpoints, their parameters, descriptions, and response formats. The agent uses the schema descriptions to decide which tool to call for a given user query and to extract the correct parameters from natural language. Well-written descriptions directly improve tool selection accuracy.

**Q: How do Bedrock Agents maintain conversation context across multiple turns?**
A: Agents use session management via `sessionId`. When multiple `invoke_agent` calls share the same session ID, the agent retains the conversation history and can resolve references to previous turns (e.g., pronouns like "it" or "that service"). A new session ID starts a fresh conversation with no history. Sessions expire after the configured `idleSessionTTLInSeconds`.

**Q: What are the required components to create a functional Bedrock Agent?**
A: A Bedrock Agent requires: (1) an IAM role granting Bedrock permission to invoke foundation models and Lambda functions, (2) a foundation model selection for reasoning, (3) agent instructions that guide behavior, (4) at least one action group with an OpenAPI schema and either a Lambda function or ROC configuration, and (5) the agent must be prepared and have an alias created before it can be invoked.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Claude Sonnet 4.5 — Agent invocations (Sections C-D, ~5 queries) | ~5 API calls, ~10K tokens | ~$0.15 |
| Claude Sonnet 4.5 — LangChain agent (Section F, optional) | ~1 API call, ~2K tokens | ~$0.03 |
| AWS Lambda — Function invocations | ~5 invocations, <1s each | < $0.01 |
| Bedrock Agent — No additional charge | Billed via model invocation costs | $0.00 |
| **Total** | | **~$2-3** |

Bedrock Agents do not have a separate service charge — you pay only for the foundation model invocations the agent makes during orchestration. Each agent turn may involve multiple model calls (for reasoning steps), so actual token usage is higher than a single direct model call. Lambda costs are negligible at this scale.